In [23]:
import requests
import pandas as pd

In [13]:
"""
Problem 1 — Find Customers by City and Segment
"""
def customers_in_city_segment(conn, city, segment):
    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql(query, conn, params=(city, segment))

In [14]:
"""
Problem 2 — Join Customers with Their Orders
"""
def orders_with_customer_names(conn):
    query = """
        SELECT c.name, o.order_date, o.amount
        FROM orders o
        INNER JOIN customers c ON o.customer_id = c.customer_id
        LIMIT 10
    """
    return pd.read_sql(query, conn)

In [15]:
"""
Problem 3 — Total Revenue per Customer (High Spenders Only)
"""
def high_spending_customers(conn, min_total=300000):
    query = """
        SELECT c.name, SUM(o.amount) as total_spent
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id
        HAVING total_spent > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql(query, conn, params=(min_total,))

In [16]:
"""
Problem 4 — Top N Customers by Spend
"""
def top_n_customers(conn, n=3):
    query = """
        SELECT c.name, SUM(o.amount) as total_spent
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql(query, conn, params=(n,))

In [17]:
"""
Problem 5 — Search Orders by Customer Name (Safely!)
"""
def find_orders_by_customer(conn, customer_name):
    query = """
        SELECT o.order_id, o.order_date, o.amount
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.name = ?
    """
    cur = conn.cursor()
    cur.execute(query, (customer_name,))
    return cur.fetchall()

In [18]:
"""
Problem 6 — Monthly Revenue Report
"""
def monthly_revenue(conn):
    query = """
        SELECT strftime('%Y-%m', order_date) as month, SUM(amount) as revenue
        FROM orders
        GROUP BY month
        ORDER BY month ASC
    """
    return pd.read_sql(query, conn)

In [19]:
"""
Problem 7 — Fetch Weather for a City
"""
def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json().get("current_weather")
    return None

get_current_weather(37.5660,126.9784)

{'time': '2026-10-11T08:45',
 'interval': 900,
 'temperature': 23.9,
 'windspeed': 1.3,
 'winddirection': 326,
 'is_day': 1,
 'weathercode': 2}

In [20]:
"""
Problem 8 — Weather for Multiple Cities into a DataFrame
"""
def weather_for_cities(cities):
    results = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        if weather:
            weather["city"] = city_name
            results.append(weather)
    return pd.DataFrame(results)

In [21]:
"""
Problem 9 — Respect the Rate Limit
"""
def weather_for_cities_paced(cities, delay_seconds=1):
    results = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        if weather:
            weather["city"] = city_name
            results.append(weather)
        import time
        time.sleep(delay_seconds)
    return pd.DataFrame(results)

In [ ]:
test_cities = [("Seoul", 37.5665, 126.9780), ("Busan", 35.1796, 129.0756), ("Tokyo", 35.6762, 139.6503)]
weather_for_cities(test_cities)
weather_for_cities_paced(test_cities)